# Reintegrate losses in your own embroidery photograph

Companion notebook for [*Digitally Reintegrating Losses in Heritage Embroidery*](https://geojenks.github.io/digitally-reintegrating-losses/) (GCH 2026).

Each masked loss is seeded with a **procedural texture init** (knot bumps, parallel satin threads, coiled purl) and restyled by **SDXL + a stitch-structure LoRA** trained on 17th-century English embroideries.

**How to use it:**
1. Draw your loss masks with the mask tool on the [project page](https://geojenks.github.io/digitally-reintegrating-losses/) and download the **job bundle** (.zip).
2. Here, choose *Runtime → Run all*. When asked, upload the bundle (plus any custom LoRA .safetensors it names). Or pick the castle sample in step 4 to try it without a bundle.
3. The result is shown at the end and downloaded as a .zip.

**Hardware:** Colab's free T4 runs the SDXL route (a few minutes per stitch type). The stronger FLUX route used for the paper's final figures needs Colab Pro (A100) or a ~24 GB GPU, plus the gated FLUX.1-dev weights (see step 5). Locally, clone the repo and follow its README; no notebook needed.

In [ ]:
#@title 1 – Check the GPU
!nvidia-smi -L

In [ ]:
#@title 2 – Get the code and dependencies (~2 min)
!git clone -q https://github.com/geojenks/digitally-reintegrating-losses
%cd digitally-reintegrating-losses
!pip -q install diffusers transformers accelerate safetensors sentencepiece protobuf opencv-python scipy peft

In [ ]:
#@title 3 – Download the SDXL stitch LoRAs (85 MB each)
import os, urllib.request
REL = "https://github.com/geojenks/digitally-reintegrating-losses/releases/latest/download"
for run in ["french_knot_stitch_sdxl_lora_v1",
            "satin_stitch_sdxl_lora_v1",
            "silk_purl_sdxl_lora_v1"]:
    os.makedirs(f"output/{run}", exist_ok=True)
    dst = f"output/{run}/{run}.safetensors"
    if not os.path.exists(dst):
        print("downloading", run)
        urllib.request.urlretrieve(f"{REL}/{run}.safetensors", dst)
print("LoRAs ready")

In [ ]:
#@title 4 – Choose the job
#@markdown Upload the .zip job bundle from the mask tool, plus any custom LoRA .safetensors files it names (select them all in one go). Or use the bundled castle sample.
SOURCE = "upload a job bundle from the mask tool"  #@param ["upload a job bundle from the mask tool", "castle sample"]
import os, json, shutil, zipfile
from pathlib import Path

if SOURCE == "castle sample":
    JOB = Path("data/jobs/castle")
else:
    from google.colab import files
    up = files.upload()
    zips = [n for n in up if n.lower().endswith(".zip")]
    assert len(zips) == 1, f"upload exactly one job bundle .zip (got {zips})"
    JOB = Path("jobs") / Path(zips[0]).stem
    shutil.rmtree(JOB, ignore_errors=True)
    JOB.mkdir(parents=True)
    zp = Path("uploads") / zips[0]
    zp.parent.mkdir(exist_ok=True)
    zp.write_bytes(up[zips[0]])
    with zipfile.ZipFile(zp) as z:
        z.extractall(JOB)
    if not (JOB / "job.json").exists():            # zip with one top folder inside
        JOB = next(JOB.glob("*/job.json")).parent
    # custom LoRAs go where the job's "lora" paths expect them (usually loras/)
    loras = {n: up[n] for n in up if n.lower().endswith(".safetensors")}
    stitches = json.loads((JOB / "job.json").read_text()).get("stitches", {})
    missing = {k: JOB / s["lora"] for k, s in stitches.items() if not (JOB / s["lora"]).exists()}
    for k, dst in list(missing.items()):
        if dst.name in loras:
            dst.parent.mkdir(parents=True, exist_ok=True)
            dst.write_bytes(loras.pop(dst.name)); del missing[k]
    if len(missing) == 1 and len(loras) == 1:      # one file, renamed on upload
        (k, dst), (n, blob) = next(iter(missing.items())), next(iter(loras.items()))
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_bytes(blob); del missing[k]
    for k, dst in missing.items():
        print(f"WARNING: no LoRA for custom stitch '{k}' (expected {dst}); the run will stop there")

JOB_NAME = json.loads((JOB / "job.json").read_text())["name"]
print("job:", JOB_NAME, "from", JOB)

In [ ]:
#@title 5 – Settings
#@markdown **variants_per_region** > 0 also saves that many alternative fills per region (shown as a grid below; compose them with `demo/region_picker.html`). Each one costs a diffusion pass.
variants_per_region = 0  #@param {type:"slider", min:0, max:8, step:1}
#@markdown **seed**: 0 keeps the job's own seed.
seed = 0  #@param {type:"integer"}
#@markdown **model**: sdxl_base runs on the free T4. flux_base needs Colab Pro (A100), the gated [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev) weights (accept the licence, log in with a Hugging Face token) and `FLUX_DEV_SAFETENSORS` set to your `flux1-dev.safetensors`. Custom LoRAs must be trained for the chosen model.
model = "sdxl_base"  #@param ["sdxl_base", "flux_base"]
LORA_VARIANT = "v1" if model == "sdxl_base" else "trigonly_v2"
if model == "flux_base":
    assert os.environ.get("FLUX_DEV_SAFETENSORS"), "set os.environ['FLUX_DEV_SAFETENSORS'] to your flux1-dev.safetensors"
    for run in ["french_knot_stitch_flux1_lora_trigonly_v2",
                "satin_stitch_flux1_lora_trigonly_v2",
                "silk_purl_flux1_lora_trigonly_v2"]:
        os.makedirs(f"output/{run}", exist_ok=True)
        dst = f"output/{run}/{run}.safetensors"
        if not os.path.exists(dst):
            print("downloading", run)
            urllib.request.urlretrieve(f"{REL}/{run}.safetensors", dst)
print("model:", model, LORA_VARIANT)

In [ ]:
#@title 6 – Reintegrate (SDXL: a few minutes per stitch type on a T4)
import subprocess, sys
OUT = Path("staged_out")
cmd = [sys.executable, "pipeline/staged_reintegrate.py", "--job", str(JOB),
       "--model", model, "--lora_variant", LORA_VARIANT, "--tries", "1", "--out", str(OUT)]
if seed:
    cmd += ["--seed", str(seed)]
if variants_per_region:
    cmd += ["--per_region", "--region_variants", str(variants_per_region)]  # variants need per-region fills
print(" ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
#@title 7 – Show the result
import glob
from IPython.display import Image as I, display
from PIL import Image
import matplotlib.pyplot as plt
RUN = Path(max(glob.glob(f"{OUT}/{JOB_NAME}/*/final.png"), key=os.path.getmtime)).parent
print(RUN)
display(I(str(RUN / "final.png"), width=700))
display(I(str(RUN / "strip.png"), width=1000))
if (RUN / "variants" / "regions.json").exists():   # one row per region, one column per variant
    rj = json.loads((RUN / "variants" / "regions.json").read_text())
    base = Image.open(RUN / "variants" / "base.png").convert("RGBA")
    regs = rj["regions"][:12]
    ncol = max(len(r["files"]) for r in regs)
    fig, axs = plt.subplots(len(regs), ncol, figsize=(2 * ncol, 2 * len(regs)), squeeze=False)
    for i, r in enumerate(regs):
        x0, y0, x1, y1 = r["bbox"]
        for j, ax in enumerate(axs[i]):
            ax.axis("off")
            if j < len(r["files"]):
                im = base.crop((x0, y0, x1, y1))
                im.alpha_composite(Image.open(RUN / "variants" / r["files"][j]).convert("RGBA"))
                ax.imshow(im); ax.set_title(r["files"][j], fontsize=7)
    plt.tight_layout(); plt.show()
    if len(rj["regions"]) > len(regs):
        print(f"showing the first {len(regs)} of {len(rj['regions'])} regions")

In [ ]:
#@title 8 – Download the output folder as a .zip
from google.colab import files
zp = shutil.make_archive(f"{JOB_NAME}_out", "zip", OUT, JOB_NAME)
files.download(zp)

## Going further

- **FLUX route (paper quality)**: pick `flux_base` in step 5 on a Colab Pro A100 (or run locally on a ~24 GB GPU). Step 5 downloads the `*_flux1_lora_trigonly_v2` LoRAs; you supply the gated FLUX.1-dev weights via `FLUX_DEV_SAFETENSORS`.
- **Region picker**: with variants on, copy `demo/region_picker.html` into the downloaded `variants/` folder and open it to compose your favourite fill per region.
- **Several jobs at once** (locally): put the bundles in one folder and run `python pipeline/run_jobs.py jobs/`. The bundle format is in `pipeline/JOB_FORMAT.md`.
- **Legacy: image + per-stitch masks**: `staged_reintegrate.py` still takes `--image photo.png --masks folder/` with black-and-white masks named `<image>__satin.png`, `<image>__french_knot.png`, `<image>__silk_purl.png` (white = the loss to fill); see the repo README.
- **Train your own stitch LoRA**: see `training/` in the repo for the exact ai-toolkit configs and `data/stitches/` for the captioning style. Add it to a job as a custom stitch (see `JOB_FORMAT.md`).